<a href="https://colab.research.google.com/github/B00921114/Practical1/blob/main/RefinedCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!pip install torch==2.0.1 transformers==4.32.0
!pip install requests numpy scikit-learn nltk schedule

In [47]:
# Import required libraries
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
import requests
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from difflib import SequenceMatcher
import schedule
import time

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [49]:
# Google Scholar search function using SerpAPI
def search_google_scholar(query, api_key, num_results=20):
    url = "https://serpapi.com/search.json"
    params = {
        "engine": "google_scholar",
        "q": query,
        "num": num_results,
        "api_key": api_key
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code} - {response.text}"

In [50]:
# Prepare the text data from search results
def prepare_text_data(results):
    texts = []
    for result in results.get("organic_results", []):
        title = result.get("title", "")
        snippet = result.get("snippet", "")
        full_text = title + " " + snippet
        cleaned_text = preprocess_text(full_text)
        texts.append(cleaned_text)
    return texts

In [53]:
# Automatic Data Refresh system
def refresh_data():
    print("Refreshing data...")
    global texts
    texts = []
    for query in queries:
        results = search_google_scholar(query, api_key, num_results=20)
        if isinstance(results, dict):
            texts += prepare_text_data(results)
    print("Data refresh complete.")

schedule.every().day.at("02:00").do(refresh_data)

Every 1 day at 02:00:00 do refresh_data() (last run: [never], next run: 2024-08-20 02:00:00)

In [54]:
# Collecting Data Using the API
api_key = "e73437c2007190db079d36557402977fc4e68a641bcc4fb9f5e43df14f18c950"  # Replace with your SerpAPI key
queries = ["machine learning in healthcare", "artificial intelligence in medicine", "deep learning in medical imaging"]
texts = []

for query in queries:
    results = search_google_scholar(query, api_key, num_results=20)
    if isinstance(results, dict):
        texts += prepare_text_data(results)


In [55]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenizing the data
input_ids = []
attention_masks = []
for text in texts:
    encoded_dict = tokenizer(
        text,
        add_special_tokens=True,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])

# Converting the lists to tensors
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor([0] * len(input_ids))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [56]:
# Creating a DataLoader and a dataloader
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(input_ids, attention_masks, labels)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [57]:
# Fine-Tune BERT with Batch Size and Epochs
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Training the loop
model.train()
for epoch in range(training_args.num_train_epochs):
    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch  # Unpack the batch
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 0, Loss: 0.8402778506278992
Epoch 0, Loss: 0.5495139956474304
Epoch 0, Loss: 0.4404168128967285
Epoch 0, Loss: 0.3366433382034302
Epoch 0, Loss: 0.24606116116046906
Epoch 0, Loss: 0.18061420321464539
Epoch 0, Loss: 0.1607969105243683
Epoch 0, Loss: 0.17326919734477997
Epoch 0, Loss: 0.15200383961200714
Epoch 0, Loss: 0.09327693283557892
Epoch 0, Loss: 0.07678922265768051
Epoch 0, Loss: 0.06323873996734619
Epoch 0, Loss: 0.05676787719130516
Epoch 0, Loss: 0.03991623967885971
Epoch 0, Loss: 0.04187962785363197


In [58]:
# Function to normalize titles
def normalize_title(title):
    title = title.lower()
    title = re.sub(r'[^a-zA-Z0-9\s]', '', title)
    return title.strip()

# Function to calculate partial match score
def partial_match_score(a, b):
    return SequenceMatcher(None, a, b).ratio()

# Function to calculate Precision@K
def precision_at_k_partial(actual, predicted, k, threshold=0.6):
    predicted_at_k = predicted[:k]
    hits = 0
    for pred in predicted_at_k:
        if any(partial_match_score(pred, act) > threshold for act in actual):
            hits += 1
    precision = hits / len(predicted_at_k)
    return precision

# Function to calculate Recall@K
def recall_at_k(actual, predicted, k, threshold=0.6):
    predicted_at_k = predicted[:k]
    hits = 0
    for act in actual:
        if any(partial_match_score(act, pred) > threshold for pred in predicted_at_k):
            hits += 1
    recall = hits / len(actual)
    return recall

# Function to calculate F1 Score
def f1_score(precision, recall):
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)

# Function to calculate Average PrecisionK
def average_precision_at_k(actual, predicted, k):
    if not actual:
        return 0.0

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:  # only count hits once
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(actual), k)

# Function to calculate the Mean Average PrecisionK
def mean_average_precision_at_k(actual_list, predicted_list, k=10):
    return np.mean([average_precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)])

# Example
actual_relevant_papers = [
    ["machine learning in healthcare a review",
     "applications of machine learning in healthcare"]
]

#  predicted_papers
predicted_papers = [normalize_title(title) for title, link in recommended_papers]

actual_relevant_papers = [[normalize_title(title) for title in actual_list] for actual_list in actual_relevant_papers]

print("Actual Relevant Papers:")
for paper in actual_relevant_papers[0]:
    print(f"- {paper}")

print("\nPredicted Papers:")
for paper in predicted_papers:
    print(f"- {paper}")

# Calculation of Precision, Recall, F1 Score, and MAP
k = 50000000
precision = precision_at_k_partial(actual_relevant_papers[0], predicted_papers, k)
recall = recall_at_k(actual_relevant_papers[0], predicted_papers, k)
f1 = f1_score(precision, recall)
map_score = mean_average_precision_at_k(actual_relevant_papers, [predicted_papers], k)

# Displaying the results
print(f"Precision@{k}: {precision:.4f}")
print(f"Recall@{k}: {recall:.4f}")
print(f"F1 Score@{k}: {f1:.4f}")
print(f"MAP@{k}: {map_score:.4f}")


Actual Relevant Papers:
- machine learning in healthcare a review
- applications of machine learning in healthcare

Predicted Papers:
- overview of deep learning in medical imaging
- deep learning in medical imaging general overview
- deep learning in medical imaging
- deep learning in medical image analysis
- an overview of deep learning in medical imaging focusing on mri
- an overview of deep learning in medical imaging
- deep learning and medical imaging
- deep learning in medical imaging and radiation therapy
- prospects of deep learning for medical imaging
- deep learning for medical image processing overview challenges and the future
- guest editorial deep learning in medical imaging overview and future promise of an exciting new technique
- stateoftheart review on deep learning in medical imaging
- a review of deep learning in medical imaging imaging traits technology trends case studies with progress highlights and future promises
- deep learning in medical imaging a brief revi

In [68]:
def get_user_query_and_recommend():
    """
    This function handles:
    1. Getting the user's query
    2. Generating recommendations based on the query
    3. Returning the list of recommended papers
    """

    # Getting the user query
    user_query = input("Enter your research topic: ")

    # Generating recommendations based on the query
    results = search_google_scholar(user_query, api_key, num_results=20)
    texts = prepare_text_data(results)
    user_profile_embedding = get_bert_embeddings([user_query], model, tokenizer)
    paper_embeddings = get_bert_embeddings(texts, model, tokenizer)
    similarities = cosine_similarity(paper_embeddings, user_profile_embedding.reshape(1, -1))
    ranked_indices = np.argsort(similarities[:, 0])[::-1]

    recommended_papers = []

    # Returning the list of recommended papers
    for i in ranked_indices[:5]:
        title = results['organic_results'][i]['title']
        link = results['organic_results'][i].get('link', 'Link not available')
        score = similarities[i][0]
        recommended_papers.append((title, link, score))

    return recommended_papers

def get_feedback(recommended_papers):
    feedback = []
    print("\nPlease rate the relevance of each recommended paper on a scale from 1 to 5 (1 = Not relevant, 5 = Very relevant):\n")
    for i, (title, link, score) in enumerate(recommended_papers, start=1):
        rating = int(input(f"Relevance of paper {i} (Title: {title}): "))
        feedback.append((title, link, score, rating))
    return feedback

def analyze_feedback(feedback):
    avg_rating = np.mean([rating for _, _, _, rating in feedback])
    print(f"\nAverage feedback rating: {avg_rating:.2f}")
    if avg_rating < 3:
        print("Feedback indicates that the recommendations may need improvement.")
    else:
        print("Feedback indicates that the recommendations are generally relevant.")

# Main loop for user interaction
def main():
    continue_interaction = True
    while continue_interaction:
        recommended_papers = get_user_query_and_recommend()
        feedback = get_feedback(recommended_papers)
        analyze_feedback(feedback)

        another_query = input("\nWould you like to enter another query? (yes/no): ").strip().lower()
        if another_query != 'yes':
            continue_interaction = False

# Running to the interaction loop
if __name__ == "__main__":
    main()


Enter your research topic: deep learning

Please rate the relevance of each recommended paper on a scale from 1 to 5 (1 = Not relevant, 5 = Very relevant):

Relevance of paper 1 (Title: The deep learning revolution): 5
Relevance of paper 2 (Title: Deep learning with PyTorch): 4
Relevance of paper 3 (Title: Applied deep learning): 3
Relevance of paper 4 (Title: Grokking deep learning): 5
Relevance of paper 5 (Title: Deep learning in agriculture: A survey): 1

Average feedback rating: 3.60
Feedback indicates that the recommendations are generally relevant.

Would you like to enter another query? (yes/no): no


In [73]:


def collaborative_filtering(user_history, all_users_history, k=5):
    user_similarities = cosine_similarity(user_history, all_users_history)
    # The following line had an extra indent causing the error
    recommendations = np.dot(user_similarities, all_users_history)

    # Selecting top-k recommendations
    recommended_papers = np.argsort(recommendations[0])[-k:][::-1]
    return recommended_papers

def hybrid_recommendation_system(user_query, user_profile_embedding, user_history, all_users_history, k=5):
    # Content-based recommendations using BERT
    recommendations = search_google_scholar(user_query, api_key, num_results=20)
    texts = prepare_text_data(recommendations)
    paper_embeddings = get_bert_embeddings(texts, model, tokenizer)
    similarities = cosine_similarity(paper_embeddings, user_profile_embedding.reshape(1, -1))
    content_based_papers = np.argsort(similarities[:, 0])[::-1][:k]

    # Collaborative filtering recommendations
    collaborative_papers = collaborative_filtering(user_history, all_users_history, k)

    # Combining both recommendations
    final_recommendations = list(set(content_based_papers) | set(collaborative_papers))

    # Fetching and display final recommendations
    final_papers = [(recommendations['organic_results'][i]['title'], recommendations['organic_results'][i]['link']) for i in final_recommendations[:k]]
    for title, link in final_papers:
        print(f"Title: {title}")
        print(f"Link: {link}")
        print("-" * 80)

# Run This:
user_query = "energy storage"
user_profile_embedding = get_bert_embeddings([user_query], model, tokenizer)
user_history = np.array([[1, 0, 0], [0, 1, 0]])
all_users_history = np.array([[1, 0, 1], [0, 1, 0]])
hybrid_recommendation_system(user_query, user_profile_embedding, user_history, all_users_history)


Title: Advanced materials for energy storage
Link: https://onlinelibrary.wiley.com/doi/abs/10.1002/adma.200903328
--------------------------------------------------------------------------------
Title: Energy storage systems—Characteristics and comparisons
Link: https://www.sciencedirect.com/science/article/pii/S1364032107000238
--------------------------------------------------------------------------------
Title: Energy storage
Link: https://escholarship.org/content/qt7dt8q203/qt7dt8q203.pdf
--------------------------------------------------------------------------------
Title: Renewable energy and energy storage systems
Link: https://www.sciencedirect.com/science/article/pii/S0360544217312306
--------------------------------------------------------------------------------
Title: Energy storage technologies: The past and the present
Link: https://ieeexplore.ieee.org/abstract/document/6922604/
--------------------------------------------------------------------------------
